In [ ]:
""" base """
dataName = "IMDBC5base"
# dataName = "IMDBCLinearbase"
# dataName = "stackn-single"
# dataName = "taxi-single"
# dataName = "Brazilnewbase"

""" class """
# dataName = "IMDBC5"
# dataName = "IMDBLargeC5"
# dataName = "Brazilnew"


""" regression """
# dataName = "IMDBCLinear"
# dataName = "taxi"
# dataName = "stackn"
# dataName = "IMDBLargeCLinear"

In [ ]:
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import copy

# Load Data

In [ ]:
DATAPATH = "/home/jiayi/disk/C-craig/dataset/"

## Judge Task

In [ ]:
def judgeTask(dataName):
    # Classification datasets
    if dataName in['IMDBC5', 'IMDBLargeC5', 'Brazilnew','Brazilnewbase']:
        return 'class'
    else:
        return 'reg'

In [ ]:
def load_dataset(dataset, prop=0.1, regression=False):
    assert dataset in ['IMDBCLinear', 'IMDBLargeCLinear', 'Brazilnew', 'IMDBC5', 'IMDBLargeC5', 
                       'taxi', 'stackn','Brazilnewbase', 
                       'IMDBC5base','IMDBCLinearbase','stackn-single','taxi-single']

    X_train = np.load(DATAPATH + "{}-train-X.npy".format(dataset))
    X_val = np.load(DATAPATH + "{}-val-X.npy".format(dataset))
    X_test = np.load(DATAPATH + "{}-test-X.npy".format(dataset))
    y_train = np.load(DATAPATH + "{}-train-y.npy".format(dataset))
    y_val = np.load(DATAPATH + "{}-val-y.npy".format(dataset))
    y_test = np.load(DATAPATH + "{}-test-y.npy".format(dataset))

    if regression == False:
        assert  dataset in ['IMDBC5','IMDBLargeC5', 'Brazilnew', 'IMDBC5base']
        print("Is Multi class")
        if dataset in ['IMDBC5', 'IMDBLargeC5', 'Brazilnew', 'IMDBC5base','Brazilnewbase']:
            num_class = 5
        print("Num class  ", num_class)
        if dataset in ['Brazil5']:
            y_train-=1
            y_val-=1
            y_test-=1
        print(np.unique(y_train))
        print(np.unique(y_val))
        print(np.unique(y_test))
        y_train = y_train.astype(np.int32)
        y_val = y_val.astype(np.int32)
        y_test = y_test.astype(np.int32)
#         y_train = np.eye(num_class)[y_train]
#         y_val = np.eye(num_class)[y_val]
#         y_test = np.eye(num_class)[y_test]
#     elif not regression:
        y_train = np.reshape(y_train, (-1, 1))
        y_val = np.reshape(y_val, (-1, 1))
        y_test = np.reshape(y_test, (-1, 1))
    print(f'Training size: {len(y_train)}, Test size: {len(y_test)}')
    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
# LoadDataset(dataName)
ifRegression = judgeTask(dataName)=='reg'
X_train, y_train, X_val, y_val, X_test, y_test = load_dataset(dataName, regression=ifRegression)

# Build DNN

In [ ]:
NUM_CLASSES_DICT ={
    "IMDBC5":5,
    "IMDBLargeC5":5,
    "Brazilnew":5,
    "IMDBC5base":5
}
if judgeTask(dataName) == 'class':
    NUM_CLASSES = NUM_CLASSES_DICT[dataName]
else:
    NUM_CLASSES = -1

In [ ]:
use_gpu = torch.cuda.is_available()
# use_gpu = False
print(use_gpu)
if use_gpu:
    torch.cuda.set_device(2)

# BatchSize = 1024
BatchSize = 8192 
# BatchSize = 169247 * 4


epochs = 20



SEED = 2020210966
learning_rate =1e-3

## Class. DNN

In [ ]:
class DNNNet(nn.Module):
    def __init__(self, input_shape=X_train.shape[1], hidden_size1=200, category=NUM_CLASSES):
        super(DNNNet, self).__init__()
        self.input_shape = input_shape
        self.hidden_size1 = hidden_size1
        #         self.hidden_size2 = hidden_size2
        self.category = category
        self.layers = nn.Sequential(
            nn.Linear(self.input_shape, self.hidden_size1),
            nn.ReLU(),

            nn.Linear(self.hidden_size1, self.hidden_size1),
            nn.Tanh(),
            nn.ReLU(),


            nn.Linear(self.hidden_size1, self.category),
            nn.Softmax(dim=-1)
        )

#         self.outlayer = nn.Softmax(dim=-1)
        self.init_params()

    def init_params(self):
        # xavier_uniform_
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if (m.bias is not None):
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):

        x = self.layers(x)

        return x



## Reg. DNN

In [ ]:
class DNNNetReg(nn.Module):
    def __init__(self, input_shape=X_train.shape[1], hidden_size1=100):
        super(DNNNetReg, self).__init__()
        self.input_shape = input_shape
        self.hidden_size1 = hidden_size1
        self.layers = nn.Sequential(
            nn.Linear(self.input_shape, self.hidden_size1),
            nn.ReLU(),

            nn.Linear(self.hidden_size1, self.hidden_size1),
            nn.ReLU(),

            nn.Linear(self.hidden_size1, 1),
            
        )

        self.init_params()

    def init_params(self):
        # xavier_uniform_
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if (m.bias is not None):
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):

        x = self.layers(x)

        return x



# train

## to Tensor

In [ ]:
X_train = torch.FloatTensor(X_train)
X_val = torch.FloatTensor(X_val)
X_test = torch.FloatTensor(X_test)

if judgeTask(dataName) == 'class':
    y_train = torch.LongTensor(y_train)
    y_val = torch.LongTensor(y_val)
    y_test = torch.LongTensor(y_test)


    y_train[y_train<0] = 0
    y_val[y_val<0]=0
    y_test[y_test<0]=0
    
    y_train = y_train.reshape(-1)
    y_val = y_val.reshape(-1)
    y_test = y_test.reshape(-1)
else:
    y_train = torch.FloatTensor(y_train).reshape(-1,1)
    y_val = torch.FloatTensor(y_val).reshape(-1,1)
    y_test = torch.FloatTensor(y_test).reshape(-1,1)
    
if use_gpu:
    X_train = X_train.cuda()
    X_val = X_val.cuda()
    X_test = X_test.cuda()
    y_train = y_train.cuda()
    y_val = y_val.cuda()
    y_test = y_test.cuda()
    

## Build Model

In [ ]:
def buildModel():
    if judgeTask(dataName) == 'class':
        model = DNNNet()
    else:
        model = DNNNetReg()
        
    # model to GPU
    if use_gpu:
        model = model.cuda()
    return model


# Funcs

## Train

In [1]:
def train(model, X_train, y_train, X_val, y_val, epochs, BatchSize, lr=learning_rate, vBatchSize=10**6):

    if judgeTask(dataName) =='class':
        criterion = nn.CrossEntropyLoss()
        isClass = True
    else:
        criterion = nn.MSELoss()
        
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=10**(-4))
    
    
    print("Trained on:")
    print("--- [{}]".format(X_train.shape))
    
    model.train()
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma = 0.98, last_epoch=-1)
    best_val_loss = 10**9
    retModel = copy.deepcopy(model)
    for epoch in range(epochs):
        optimizer.zero_grad()

        num_itr = X_train.shape[0]//BatchSize
        for i in range(num_itr):
            st = i * BatchSize
            en = (i + 1) * BatchSize
            if en > X_train.shape[0]:
                en = X_train.shape[0]

            itrData = X_train[st:en]
            itrLabel = y_train[st:en]

            outputs = model(itrData)

#             print('see outputs', outputs.shape)
#             print('see itrLabel', itrLabel.shape)
            loss = criterion(outputs, itrLabel)
#             print('see loss is ', loss)
#             print(type(loss))
            loss.backward()
            optimizer.step()
        
        """ Validate """
        with torch.no_grad():
            val_st = 0
            val_predict_list = []
            while val_st < X_val.shape[0]:

                val_en = val_st + vBatchSize
                if val_en > X_val.shape[0]:
                    val_en = X_val.shape[0]
                val_predict_list.append(model(X_val[val_st:val_en,:]))
                val_st = val_st + vBatchSize

            val_predict = torch.vstack(val_predict_list)
            val_loss = criterion(val_predict, y_val)
            if val_loss < best_val_loss:
                print("Current epoch is [{}], use this better model".format(epoch))
                best_val_loss = val_loss
                retModel = copy.deepcopy(model)
        scheduler.step()
    return retModel

NameError: name 'learning_rate' is not defined

## test

In [ ]:
def test(model, tBatchSize=10**6):
    model.eval()
    
    if judgeTask(dataName) =='class':
        criterion = nn.CrossEntropyLoss()
    else:
        criterion = nn.MSELoss()
        
    with torch.no_grad():
        test_st = 0
        test_predict_list = []
        while test_st < X_test.shape[0]:

            test_en = test_st + tBatchSize
            if test_en > X_test.shape[0]:
                test_en = X_test.shape[0]
            test_predict_list.append(model(X_test[test_st:test_en,:]))
            test_st = test_st + tBatchSize

        pred = torch.vstack(test_predict_list)

        
    print("Prediction result shape is ", pred.shape)

    
    # Evaluation
    print("Test Loss:", criterion(pred, y_test))
    if judgeTask(dataName) =='class':
        pred = pred.detach().max(axis=1)[1]

    return pred

In [ ]:
def test_not_use(model):
    global X_test,y_test
    this_X_test = X_test.detach().clone()
    this_y_test = y_test.detach().clone()
    
    model.eval()
    
    criterion = nn.CrossEntropyLoss()
    if this_X_test.shape[0] > 10**6:
        this_X_test = this_X_test[:10**6]
        this_y_test = this_y_test[:10**6]

    with torch.no_grad():
        pred = model(this_X_test)
    print("Prediction result shape is ", pred.shape)

    
    # Evaluation
    print("Test Loss:", criterion(pred, this_y_test))
    pred = pred.detach().max(axis=1)[1]

    result = 1.0 * (pred == this_y_test)

    print("Test Accuracy:", result.mean())
    
    return pred

## Evaluate Performance

In [ ]:
from sklearn import metrics

In [ ]:
def EvaluateClass(pred, y_test):
    predictions = pred.cpu().detach()
    y_test = y_test.cpu().detach()
    test_predict_y = predictions

    full_test_f1 = metrics.f1_score(y_test, test_predict_y , average = 'micro')
    full_test_f1_weighted = metrics.f1_score(y_test, test_predict_y , average = 'weighted')
    full_test_recall = metrics.recall_score(y_test, test_predict_y , average = 'weighted')
    full_test_recall_weighted = metrics.recall_score(y_test, test_predict_y , average = 'weighted')

    full_test_accuracy = metrics.accuracy_score(y_test, test_predict_y)

    print('F1-score Mircro   [{}]'.format(full_test_f1))
    print('F1-score Weighted [{}]'.format(full_test_f1_weighted))
    print('Recall            [{}]'.format(full_test_recall))
    print("Accuracy          [{}]".format(full_test_accuracy))
    
    return full_test_accuracy

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

In [ ]:
def EvaluateReg(pred, y_test):
    predictions = pred.cpu().detach()
    y_test = y_test.cpu().detach()
    
    print("Mean Absolute    Error : " + str(mean_absolute_error(predictions, y_test)))
    print("Mean Squared     Error : " + str(mean_squared_error(predictions, y_test)))
    RMSE = np.sqrt(mean_squared_error(predictions, y_test))
    print("Root Mean Squared     Error : " + str(RMSE))
    return RMSE

In [ ]:
def Evaluate(pred, y_test):
    if judgeTask(dataName) == 'class':
        return EvaluateClass(pred, y_test)
    else:
        return EvaluateReg(pred, y_test)

## trainWithCoreset 

In [ ]:
def trainWithCoreset(model, X_train, W_train, y_train, X_val, y_val, epochs, BatchSize, 
                     lr=learning_rate,vBatchSize=10**6, verbose=True):

    if judgeTask(dataName) =='class':
        criterion = nn.CrossEntropyLoss(reduction='none')
    else:
        criterion = nn.MSELoss(reduction='none')
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=10**(-4))
    
    if verbose:
        print("Trained on:")
        print("--- [{}]".format(X_train.shape))

    model.train()
    
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma = 0.98, last_epoch=-1)
    retModel = copy.deepcopy(model)
    best_val_loss = 10**9
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        
        num_itr = X_train.shape[0]//BatchSize

        for i in range(num_itr):
            st = i * BatchSize
            en = (i + 1) * BatchSize
            if en > X_train.shape[0]:
                en = X_train.shape[0]

            itrData = X_train[st:en]
            itrLabel = y_train[st:en]

            outputs = model(itrData)

            loss = (criterion(outputs, itrLabel) * W_train[st:en]).mean()

            loss.backward()
            optimizer.step()
    
        """ Validate """
        with torch.no_grad():
            val_st = 0
            val_predict_list = []
            while val_st < X_val.shape[0]:

                val_en = val_st + vBatchSize
                if val_en > X_val.shape[0]:
                    val_en = X_val.shape[0]
                val_predict_list.append(model(X_val[val_st:val_en,:]))
                val_st = val_st + vBatchSize

            val_predict = torch.vstack(val_predict_list)
            val_loss = criterion(val_predict, y_val).mean()
#             print('see val_loss')
#             print("val")
            if val_loss < best_val_loss and verbose==True:
                print("Current epoch is [{}], use this better model".format(epoch))
                best_val_loss = val_loss
                retModel = copy.deepcopy(model)
        scheduler.step()
    return retModel

## Load coreset

In [ ]:
def LoadCoreset(coreset_from, data, subset_size, batch=0,sampleSize=0):
    if coreset_from == 'scratch':
        file_name = ''

    elif coreset_from == 'diskCRAIG':
        file_name = '/home/jiayi/disk/C-craig/inuse/{}-{}.npz'.format(data, subset_size)
    elif coreset_from == 'diskLTLG':
        file_name = '/home/jiayi/disk/C-craig/inuse/{}-{}-ltlg.npz'.format(data, subset_size)
    elif coreset_from == 'diskOurs':
        if batch==0:
            if subset_size == 0.00001:
                file_name = '/home/jiayi/disk/C-craig/inuse/{}-0.00001-ours.npz'.format(data)
            else:
                file_name = '/home/jiayi/disk/C-craig/inuse/{}-{}-ours.npz'.format(data, str(subset_size))
        elif batch==1: 
            file_name = '/home/jiayi/disk/C-craig/BatchGen/{}-{}-coreset.npz'.format(data, str(subset_size))
        elif batch==2: 
            assert sampleSize !=0
            file_name = '/home/jiayi/disk/C-craig/SampleSize/{}-{}-{}-coreset.npz'.format(data, str(subset_size),sampleSize )
        else:
            assert False
    print("【Load file path】 is ", file_name)


    if file_name != '':
        print(f'reading from {file_name}')
        dataset = np.load(f'{file_name}')
        order, weights, total_ordering_time = dataset['order'], dataset['weight'], dataset['order_time']
        print(" 【Coreset size】 is ", order.shape)
        return order, weights, total_ordering_time

# Run

## Full Data

In [ ]:
%%time
model = buildModel()

if dataName =='stackn':
    model = train(model, X_train, y_train, X_val, y_val, epochs=50, BatchSize=10000, lr=0.0001)
elif dataName =='taxi':
    model = train(model, X_train, y_train, X_val, y_val, epochs=100, BatchSize=10000, lr=0.00001)
elif dataName =='IMDBCLinear':
    model = train(model, X_train, y_train, X_val, y_val, epochs=100, BatchSize=1000, lr=0.0001)
elif dataName =='IMDBLargeCLinear':
    model = train(model, X_train, y_train, X_val, y_val, epochs=100, BatchSize=100000, lr=0.0001)
elif dataName =='IMDBC5':
    model = train(model, X_train, y_train, X_val, y_val, epochs=100, BatchSize=1000, lr=0.001)
elif dataName =='Brazilnew':
    model = train(model, X_train, y_train, X_val, y_val, epochs=400, BatchSize=100, lr=0.0001)
else:
    model = train(model, X_train, y_train, X_val, y_val, epochs, BatchSize)

In [ ]:
pred = test(model)

In [ ]:
full_result = Evaluate(pred, y_test)

## Sample

In [ ]:
SAMPLE_PROP_DICT = {
    "IMDBC5":0.0128,
    "IMDBLargeC5":0.0016,
    "Brazilnew":0.0016,
    "IMDBCLinear": 0.0032,
    "IMDBLargeCLinear": 0.0016,
    "stackn":0.0032,
    "taxi":0.0032
}

In [ ]:
PROP = SAMPLE_PROP_DICT[dataName]

In [ ]:
idxs = np.arange(X_train.shape[0])
#rng = np.random.RandomState(1234)
np.random.shuffle(idxs)
# idxs = torch.Tensor(idxs)
# if use_gpus:
#     idxs = idxs.cuda()
print(idxs)

In [ ]:
idxs = idxs[:(int)(PROP * X_train.shape[0])]

In [ ]:
# SP_X_train = X_train[:(int)(PROP * X_train.shape[0])]
# SP_y_train = y_train[:(int)(PROP * y_train.shape[0])]
SP_X_train = X_train[idxs]
SP_y_train = y_train[idxs]

In [ ]:
SPmodel = buildModel()

if dataName == 'IMDBC5':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=200, BatchSize=70, lr=0.001)
elif dataName =='Brazilnew':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=400, BatchSize=10, lr=0.0001)
elif dataName =='IMDBLargeC5':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=10, BatchSize=5000, lr=0.0001)
    

elif dataName =='IMDBCLinear':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=200, BatchSize=10, lr=0.001)
elif dataName =='stackn':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=20, BatchSize=10, lr=0.0001)
elif dataName =='taxi':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=200, BatchSize=100, lr=0.0001)
elif dataName =='IMDBLargeCLinear':
    SPmodel = train(SPmodel, SP_X_train, SP_y_train, X_val, y_val, epochs=100, BatchSize=1000, lr=0.001)

In [ ]:
SPpred = test(SPmodel)
sample_result = Evaluate(SPpred, y_test)

## Coreset

In [ ]:
PROP = SAMPLE_PROP_DICT[dataName]

In [ ]:

order, weights, _ = LoadCoreset('diskOurs', dataName, subset_size=PROP)

In [ ]:
CS_X_train = X_train[order,:]
CS_W_train = torch.Tensor(weights)
if use_gpu:
    CS_W_train = CS_W_train.cuda()
CS_y_train = y_train[order]


In [ ]:

idxs = np.arange(CS_X_train.shape[0])
np.random.shuffle(idxs)
CS_X_train = CS_X_train[idxs]
CS_y_train = CS_y_train[idxs]
CS_W_train = CS_W_train[idxs]

In [ ]:
%%time
CSmodel = buildModel()
if dataName == 'IMDBC5':
#     CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=200, BatchSize=70, lr=0.0001)
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=500, BatchSize=50, lr=0.0001)
elif dataName =='Brazilnew':
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=400, BatchSize=20, lr=0.001)
if dataName == 'IMDBLargeC5':
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=200, BatchSize=1000, lr=0.0001)

elif dataName =='IMDBCLinear':
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=200, BatchSize=10, lr=0.00001)
elif dataName =='stackn':
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=200, BatchSize=200, lr=0.00001)
#     CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=1000, BatchSize=10000, lr=0.0001)
elif dataName =='taxi':
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=100, BatchSize=20, lr=0.0001)
elif dataName =='IMDBLargeCLinear':
    CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=200, BatchSize=1000, lr=0.00001)


In [ ]:
CSpred = test(CSmodel)
coreset_result = Evaluate(CSpred, y_test)

## Hyper

In [ ]:
batch_list = [10, 20, 50, 100, 200, 500, 1000, 2000]
lr_list = [0.005, 0.001, 0.0005, 0.0001, 0.00005, 0.00001]
MIN_res = 999
for batchSize in batch_list:
    for LR in lr_list:
        CSmodel = buildModel()
        CSmodel = trainWithCoreset(CSmodel, CS_X_train, CS_W_train, CS_y_train, X_val, y_val, epochs=500, 
                                   BatchSize=batchSize, lr=LR)
        CSpred = test(CSmodel)
        coreset_result = Evaluate(CSpred, y_test)
        if coreset_result < MIN_res:
            MIN_res = coreset_result
            print("# New better! BSize {} LR {}    new MIN = [{}]".format(batchSize, LR, MIN_res))